# Ejemplo: Red Neuronal con JAX + Flax

Este notebook demuestra cómo entrenar una red neuronal simple para clasificación usando JAX con Flax.

In [ ]:
import jax
import jax.numpy as jnp
from flax import linen as nn
from flax.training import train_state
import optax
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

print(f"JAX version: {jax.__version__}")
print(f"Devices: {jax.devices()}")

## 1. Preparar datos

In [ ]:
X, y = make_classification(
    n_samples=1000, n_features=20, n_informative=15,
    n_classes=3, random_state=42
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Convertir a arrays de JAX
X_train = jnp.array(X_train, dtype=jnp.float32)
y_train = jnp.array(y_train, dtype=jnp.int32)
X_test = jnp.array(X_test, dtype=jnp.float32)
y_test = jnp.array(y_test, dtype=jnp.int32)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")

## 2. Definir modelo

In [ ]:
class SimpleNN(nn.Module):
    @nn.compact
    def __call__(self, x, training: bool = True):
        x = nn.Dense(64)(x)
        x = nn.relu(x)
        x = nn.Dropout(0.3, deterministic=not training)(x)
        x = nn.Dense(32)(x)
        x = nn.relu(x)
        x = nn.Dense(3)(x)
        return x

model = SimpleNN()

## 3. Crear TrainState

In [ ]:
def create_train_state(key):
    params = model.init(key, X_train[:1], training=False)
    tx = optax.adam(1e-3)
    return train_state.TrainState.create(
        apply_fn=model.apply,
        params=params,
        tx=tx
    )

key = jax.random.key(42)
state = create_train_state(key)

## 4. Definir funciones de entrenamiento

In [ ]:
@jax.jit
def train_step(state, batch):
    x, y = batch
    
    def loss_fn(params):
        logits = state.apply_fn(params, x, training=True)
        return jnp.mean(
            optax.softmax_cross_entropy_with_integer_labels(
                logits, y
            )
        )
    
    grad_fn = jax.value_and_grad(loss_fn)
    loss, grads = grad_fn(state.params)
    state = state.apply_gradients(grads=grads)
    return state, loss

@jax.jit
def compute_accuracy(params, x, y):
    logits = state.apply_fn(params, x, training=False)
    predictions = jnp.argmax(logits, axis=-1)
    return jnp.mean(predictions == y)

## 5. Entrenar

In [ ]:
epochs = 20
batch_size = 32

for epoch in range(epochs):
    # Shuffle data
    perm = jax.random.permutation(key, len(X_train))
    X_shuffled = X_train[perm]
    y_shuffled = y_train[perm]
    
    # Train batches
    total_loss = 0
    for i in range(0, len(X_train), batch_size):
        batch_x = X_shuffled[i:i+batch_size]
        batch_y = y_shuffled[i:i+batch_size]
        state, loss = train_step(state, (batch_x, batch_y))
        total_loss += loss
    
    if (epoch + 1) % 5 == 0:
        acc = compute_accuracy(state.params, X_test, y_test)
        print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss:.4f}, Acc: {acc:.4f}")

## 6. Guardar parámetros

In [ ]:
# Guardar parámetros (no el estado completo)
import pickle
with open('model_jax_params.pkl', 'wb') as f:
    pickle.dump(state.params, f)
print("Parámetros guardados en model_jax_params.pkl")